In [ ]:
import sys
sys.path.append("..")


In [5]:

import shared_libraries.data_processing_utils as processing
import pandas as pd
import pickle

from typing import Dict, List
from fastf1.core import Session

In [ ]:
with open("../ig_sessions.pickle", "rb") as file:
    sessions: List[Session] = pickle.load(file)

In [ ]:
compounds_map = pd.read_csv("../ig_compounds_map.csv")

In [8]:
# Create queue of compound data
compd_q = []
for idx in compounds_map.index:
    compd_q.append(compounds_map.loc[idx, :])
compd_q.reverse()

# Merge data from queue with sessions
sess_n_compds = []
for s in sessions:
    s_info = s.session_info
    while True:
        compds = compd_q.pop()
        year_matches = compds["year"] == s_info["StartDate"].year
        name_matches = compds["gp"] == s_info["Meeting"]["Name"]
        if year_matches and name_matches:
            break
    sess_n_compds.append((s, compds))

In [9]:
sess_compds_by_circuit = {}
for s, cpds in sess_n_compds:
    circuit = s.session_info["Meeting"]["Circuit"]["ShortName"]
    if circuit not in sess_compds_by_circuit.keys():
        sess_compds_by_circuit[circuit] = []
    sess_compds_by_circuit[circuit].append((s, cpds))


In [10]:
# Create raw lap data, still split by session
raw_data_by_circuit_split: Dict[str, List[pd.DataFrame]] = {}
for c, ss_n_cs in sess_compds_by_circuit.items():
    raw_data_by_circuit_split[c] = []
    for s, cmpds in ss_n_cs:
        # Get compound mapping
        mapping = cmpds.loc[["soft", "medium", "hard"]]
        # Extract lap data from sessions
        lap_data = processing.get_lap_data_with_weather(s)
        fitted_mapping = pd.concat([pd.DataFrame(mapping).T] * lap_data.shape[0], ignore_index=True)
        # Join lap data with compound data
        lap_data_with_cpds = pd.concat([lap_data, fitted_mapping], axis="columns")
        raw_data_by_circuit_split[c].append(lap_data_with_cpds)


In [11]:

for c, dfs in raw_data_by_circuit_split.items():
    for df in dfs:
        # Add lap time z-score for each lap, calculated locally for each session and 
        processing.add_z_score_for_laps(df)
        # Add IsPitLap column de
        processing.add_is_pit_lap(df)

In [12]:
# Concatenate laps for each circuit
raw_data_by_circuit: Dict[str, pd.DataFrame] = {}
for c, dfs in raw_data_by_circuit_split.items():
    raw_data_by_circuit[c] = pd.concat(dfs, axis="index").reset_index()

In [13]:
# Create a "RealCompund" column which contains the actual compound (C1, C2, ..., or C5) used
for df in raw_data_by_circuit.values():
    df["RealCompound"] = None
    for idx in df.index:
        if df.loc[idx, "Compound"] in ("HARD", "MEDIUM", "SOFT"):
            df.loc[idx, "RealCompound"] = df.loc[idx, df.loc[idx, "Compound"].lower()] # type: ignore
        else:
            df.loc[idx, "RealCompound"] = df.loc[idx, "Compound"]
        



In [14]:
selected_columns = [
    "LapTimeZScore",
    "IsPitLap",
    "Compound",
    "RealCompound",
    "TyreLife",
    "FreshTyre",
    "LapNumber",
    "AirTemp",
    "Humidity",
    "Pressure",
    "Rainfall",
    "TrackTemp",
    "WindSpeed",
    "WindDirection"
]
# Clean data for each circuit
clean_data_by_circuit: Dict[str, pd.DataFrame] = {}
for c, df in raw_data_by_circuit.items():
    cleaned = df.loc[:, selected_columns].convert_dtypes().dropna()
    clean_data_by_circuit[c] = cleaned

In [ ]:
with open("../ig_data/clean_data.pickle", "wb") as file:
    pickle.dump(clean_data_by_circuit, file)